# 전체 흐름 한 번에 보기


1. 환경 설정과 라이브러리 불러오기
2. 데이터 로드
3. feature engineering 함수 정의
4. 튜닝용 feature 생성
5. Optuna로 CatBoost 하이퍼파라미터 튜닝
6. 5-fold 교차검증 초기화
7. Fold 1 ~ 5 개별 학습
8. 최종 성능 평가
9. submission 파일 생성

## 셀 1 — Import + 경로 설정

In [1]:
import os, json, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = r"Desktop\cuaida\\"
SAVE_DIR = r"Desktop\cuaida\saves\\"
os.makedirs(SAVE_DIR, exist_ok=True)
print("✓ Import 완료")

✓ Import 완료


1. 환경 설정 및 라이브러리 불러오기

맨 처음에는 pandas, numpy, CatBoost, StratifiedKFold, average_precision_score과 같은 라이브러리를 불러오고, 데이터 경로와 저장 경로를 설정합니다.

여기서는 이후 모델 학습과 평가, 파일 저장을 위한 기본 환경을 준비하는 단계라고 보면 됩니다.

* pandas, numpy : 데이터 처리
* StratifiedKFold : 교차검증
* average_precision_score : 평가 지표 계산
* CatBoostClassifier : 최종 분류 모델
* optuna : 하이퍼파라미터 튜닝
* os, json, warnings : 파일 저장/불러오기 및 경고 제어

## 셀 2 — 데이터 로드

In [2]:
train = pd.read_csv(DATA_DIR + "train.csv")
test  = pd.read_csv(DATA_DIR + "test.csv")
sub   = pd.read_csv(DATA_DIR + "sample_submission.csv")

print(f"train: {train.shape},  test: {test.shape}")
print(f"fraud 비율: {train['fraud_bool'].mean():.4f}  ({train['fraud_bool'].sum():,}건)")

train_df  = train.copy()
test_df   = test.copy()
pos_ratio = (train_df["fraud_bool"] == 0).sum() / (train_df["fraud_bool"] == 1).sum()
print(f"클래스 불균형 (0:1) = {pos_ratio:.1f}:1")

train: (700000, 33),  test: (300000, 32)
fraud 비율: 0.0109  (7,642건)
클래스 불균형 (0:1) = 90.6:1


2. 데이터 불러오기

그 다음에는 train.csv, test.csv, sample_submission.csv를 불러옵니다.

이 단계에서는 데이터 크기와 fraud 비율도 확인해서 데이터가 제대로 로드되었는지와 클래스 불균형 여부를 먼저 점검합니다.

여기서 fraud란 사기 여부를 나타내는 타겟 변수입니다.

fraud_bool = 1 -> 사기 거래
fraud_bool = 0 -> 정상 거래

즉, 이 데이터는 이진 분류 문제이고 모델의 목표는 "이 거래가 사기인지 아닌지 예측하는 것"입니다

## 셀 3 — Feature Engineering
★ 그룹 집계 피처 추가 + credit_risk_score 원본 유지

In [ ]:
def build_te_map(X_part, y_part, smooth=30):
    cat_te_cols = ["payment_type","employment_status","housing_status",
                   "device_os","application_source"]
                   
    te_map, tmp = {}, X_part.copy()
    tmp["_t"] = y_part.values
    gm = tmp["_t"].mean()
    te_map["__gm__"] = gm
    for col in cat_te_cols:
        if col in tmp.columns:
            s = tmp.groupby(col)["_t"].agg(["mean","count"])
            te_map[col] = ((s["count"]*s["mean"]+smooth*gm)/(s["count"]+smooth)).to_dict()
    return te_map


def make_features(df, te_map=None, grp_stats=None):
    df = df.copy()
    for c in ["prev_addr_months","curr_addr_months","bank_months_count"]:
        df[c] = df[c].clip(lower=0)

    # [A] 주소
    df["addr_curr_short"]      = (df["curr_addr_months"] < 6).astype(int)
    df["addr_prev_short"]      = (df["prev_addr_months"] < 6).astype(int)
    df["addr_both_short"]      = (df["addr_curr_short"] & df["addr_prev_short"]).astype(int)
    df["addr_total"]           = df["curr_addr_months"] + df["prev_addr_months"]
    df["addr_ratio"]           = df["curr_addr_months"] / (df["prev_addr_months"] + 1)

    # [B] 요청 속도
    df["req_sum"]              = df["req_rate_6h"] + df["req_rate_24h"] + df["req_rate_4w"]
    df["burst_6h_24h"]         = df["req_rate_6h"]  / (df["req_rate_24h"] + 1e-5)
    df["burst_24h_4w"]         = df["req_rate_24h"] / (df["req_rate_4w"]  + 1e-5)
    df["burst_6h_4w"]          = df["req_rate_6h"]  / (df["req_rate_4w"]  + 1e-5)
    df["burst_flag"]           = (df["burst_6h_24h"] > 3).astype(int)
    df["area_density"]         = df["zip_req_count_4w"] * df["branch_req_count_8w"]
    df["area_sum"]             = df["zip_req_count_4w"] + df["branch_req_count_8w"]
    df["req_concentration"]    = df["req_rate_6h"] / (df["req_sum"] + 1e-5)

    # [C] 신원/연락처
    df["contact_sum"]          = df["is_home_phone_valid"] + df["is_mobile_valid"]
    df["no_contact"]           = (df["contact_sum"] == 0).astype(int)
    df["name_email_low"]       = (df["name_email_sim"] < 0.3).astype(int)
    df["name_email_risk"]      = (1 - df["name_email_sim"]) * (df["is_free_email"] + 1)
    df["email_identity_risk"]  = df["is_free_email"] * (1 - df["name_email_sim"])

    # [D] 기기
    df["device_fraud"]         = (df["device_fraud_history"] > 0).astype(int)
    df["device_multi"]         = (df["device_email_cnt_8w"] > 3).astype(int)
    df["device_risk"]          = df["device_email_cnt_8w"] * (df["device_fraud_history"] + 1)
    df["dob_multi"]            = (df["dob_email_count_4w"] > 2).astype(int)
    df["device_x_email"]       = df["device_fraud_history"] * df["device_email_cnt_8w"]

    # [E] 금융 — credit_risk_score 원본 유지 (로그 변환 안 함)
    df["credit_income_r"]      = df["req_credit_limit"]  / (df["yearly_income"] + 1e-5)
    df["transfer_income_r"]    = df["init_transfer_amt"] / (df["yearly_income"] + 1e-5)
    df["zero_transfer"]        = (df["init_transfer_amt"] == 0).astype(int)
    df["credit_score_raw"]     = df["credit_risk_score"]   # ★ 원본값 그대로
    df["credit_risk_high"]     = (df["credit_risk_score"] > 600).astype(int)
    df["high_risk_low_income"] = (
        (df["credit_risk_score"] > df["credit_risk_score"].quantile(0.8)) &
        (df["yearly_income"] < 0.3)
    ).astype(int)
    # ★ credit_risk_score 구간 세분화
    df["credit_high_900"]      = (df["credit_risk_score"] > 900).astype(int)
    df["credit_high_800"]      = (df["credit_risk_score"] > 800).astype(int)
    df["credit_high_700"]      = (df["credit_risk_score"] > 700).astype(int)
    df["credit_bin"]           = pd.cut(df["credit_risk_score"],
                                          bins=[0,300,500,700,800,900,10000],
                                          labels=[0,1,2,3,4,5]).astype(float)

    # [F] 세션
    df["session_short"]        = (df["session_length_min"] < 1).astype(int)
    df["session_extreme"]      = ((df["session_length_min"] < 0.5) | (df["session_length_min"] > 120)).astype(int)
    df["no_persist"]           = (df["is_session_persistent"] == 0).astype(int)
    df["foreign_burst"]        = df["is_foreign_req"] * df["burst_flag"]
    df["foreign_anon"]         = df["is_foreign_req"] * df["is_free_email"] * df["no_persist"]

    # [G] 계좌
    df["new_bank"]             = (df["bank_months_count"] < 3).astype(int)
    df["very_new_bank"]        = (df["bank_months_count"] < 1).astype(int)
    df["no_card_new_bank"]     = ((df["has_other_cards"] == 0) & (df["bank_months_count"] < 6)).astype(int)

    # [H] 월별
    df["month_sin"]            = np.sin(2 * np.pi * df["month_idx"] / 8)
    df["month_cos"]            = np.cos(2 * np.pi * df["month_idx"] / 8)

    # [I] 복합 위험 스코어
    flags = [
        "addr_curr_short","addr_both_short","burst_flag","no_contact",
        "name_email_low","device_fraud","device_multi","dob_multi",
        "zero_transfer","credit_risk_high","session_short","no_persist",
        "is_free_email","is_foreign_req","new_bank","very_new_bank",
        "no_card_new_bank","session_extreme","high_risk_low_income",
        "credit_high_800","credit_high_900",
    ]
    df["risk_score"]           = df[[c for c in flags if c in df.columns]].sum(axis=1)
    df["risk_tier"]            = pd.cut(df["risk_score"],
                                          bins=[-1,2,4,7,100], labels=[0,1,2,3]).astype(float)

    # [J] 상호작용
    df["risk_x_burst"]         = df["risk_score"]      * df["burst_6h_24h"]
    df["device_x_addr"]        = df["device_risk"]     * df["addr_curr_short"]
    df["credit_x_burst"]       = df["credit_income_r"] * df["burst_flag"]
    df["triple_danger"]        = df["device_fraud"]    * df["burst_flag"] * df["no_contact"]
    df["high_risk_combo"]      = (df["risk_score"] >= 5).astype(int)

    # ★ [M] 그룹 집계 피처 (핵심 추가!)
    # fold leakage 방지: grp_stats를 외부에서 주입
    if grp_stats is not None:
        for key, stat in grp_stats.items():
            df[key] = df[stat["grp_col"]].map(stat["map"]).fillna(stat["global"])
    else:
        # test 시에도 grp_stats 없으면 global mean으로 fillna
        grp_defs = [
            ("employment_status", "credit_risk_score"),
            ("housing_status",    "credit_risk_score"),
            ("device_os",         "credit_risk_score"),
            ("employment_status", "req_rate_4w"),
            ("housing_status",    "req_rate_4w"),
            ("age_bucket",        "yearly_income"),
            ("employment_status", "yearly_income"),
        ]
        for grp_col, val_col in grp_defs:
            key = f"{val_col}_mean_{grp_col}"
            df[key] = df[val_col]   # fallback: 원본값

    # grp_stats 기반 편차 피처
    for grp_col, val_col in [
        ("employment_status", "credit_risk_score"),
        ("housing_status",    "credit_risk_score"),
        ("device_os",         "credit_risk_score"),
        ("employment_status", "req_rate_4w"),
        ("housing_status",    "req_rate_4w"),
    ]:
        mean_key = f"{val_col}_mean_{grp_col}"
        diff_key = f"{val_col}_diff_{grp_col}"
        if mean_key in df.columns:
            df[diff_key] = df[val_col] - df[mean_key]

    # [K] Target Encoding
    if te_map is not None:
        gm = te_map.get("__gm__", 0.011)
        for col in ["payment_type","employment_status","housing_status",
                    "device_os","application_source"]:
            if col in df.columns and col in te_map:
                df[col+"_te"] = df[col].map(te_map[col]).fillna(gm)

    # [L] 로그 변환 (credit_score_raw 제외!)
    for col in ["yearly_income","init_transfer_amt","req_credit_limit",
                "session_length_min","bank_months_count",
                "prev_addr_months","curr_addr_months",
                "zip_req_count_4w","branch_req_count_8w","device_email_cnt_8w",
                "credit_income_r","transfer_income_r",
                "area_density","device_risk","burst_6h_4w",
                "risk_x_burst","device_x_email"]:
        if col in df.columns:
            df[col] = np.log1p(df[col].clip(lower=0))

    return df


def build_grp_stats(tr_raw):
    """fold train split 기준으로만 그룹 통계 계산 (leakage 방지)"""
    grp_defs = [
        ("employment_status", "credit_risk_score"),
        ("housing_status",    "credit_risk_score"),
        ("device_os",         "credit_risk_score"),
        ("employment_status", "req_rate_4w"),
        ("housing_status",    "req_rate_4w"),
        ("age_bucket",        "yearly_income"),
        ("employment_status", "yearly_income"),
    ]
    grp_stats = {}
    for grp_col, val_col in grp_defs:
        key = f"{val_col}_mean_{grp_col}"
        mp  = tr_raw.groupby(grp_col)[val_col].mean().to_dict()
        glb = tr_raw[val_col].mean()
        grp_stats[key] = {"grp_col": grp_col, "map": mp, "global": glb}
    return grp_stats

print("✓ Feature Engineering 완료")

✓ Feature Engineering 완료


3. Feature Engineering
현재 기존 train의 경우 32개의 column으로 이루어진 data였습니다.
하지만 모델을 학습 할 때 기존 원본 변수만 쓰는게 아니라, 사기 탐지에 도움이 될 수 있도록 새로운 변수를 많이 만들어서 사용했습니다.

대표적으로는
* 주소 관련 피처
 - 현재 주소 / 이전 주소 유지 기간
 - 주소 유지 기간 비율
* 요청 빈도 / 집중도 피처
 - 최근 6시간, 24시간, 4주 요청량 조합
 - burst 여부
* 신원 / 연락처 관련 피처
 - 연락처 유효성
 - 이름,이메일 유사도 기반 위험도
* 기기 관련 피처
 - 기기 사기 이력
 - 한 기기에서 여러 이메일 사용 여부
* 금융 관련 피처
 - 신용위험점수
 - 소득 대비 한도 / 이체금액 비율
* 세션 / 계좌 / 월별 패턴 피처
* 복합 위험 점수
 - 여러 위험 신호를 합쳐 risk_score 생성



### build_te_map()

이 함수는 Target Encoding용 매핑값을 만드는 함수입니다.

적용 대상 범주형 변수는 payment_type, employment_status, housing_status, device_os, application_source

이 함수는 각 범주값별로 fraud 평균과 count를 구한 뒤 전체 평균과 함께 smooting을 적용해서 보다 ㅏㄴ정적인 target encoding 값을 만듭니다.

즉, 단순히 범주를 숫자로 바꾸는 것이 아닌, 이 범주가 과거 fraud와 얼마나 연관되어 있었는가를 반영한 수치로 변환합니다.

왜 필요한가?
 - 범주형 변수는 문자열 그대로는 모델이 잘 활용하기 어렵습니다. 특히 fraud detection에서는 단순 one-hot보다 해당 범주의 위험 경향을 직접 반영하는 방식이 더 유용할 수도 있습니다

예를 들어
 - 특정 payment type에서 fraud 비율이 높았다면 그 범주는 상대적으로 더 높은 값으로 encoding됩니다.

### build_grp-states()

이 함수는 그룹 평균 통계 피처를 만들기 위한 사전 작업을 합ㄴ디ㅏ

예를 들어 다음과 같은 조합의 평균을 계산합니다.
employment_status별 평균 credit_risk_score
housing_status별 평균 credit_risk_score
device_os별 평균 credit_risk_score
employment_status별 평균 req_rate_4w
housing_status별 평균 req_rate_4w
age_bucket별 평균 yearly_income
employment_status별 평균 yearly_income
이 평균값들을 map 형태로 저장해두었다가, 이후 실제 feature 생성 시 데이터에 붙입니다

왜 필요한가?
 - 개별 값만 보는 것보다 "이 값이 속한 그룹의 평균적인 특성"을 같이 보는 것이 fraud 탐지에 도움이 됩니다

예를 들어
 - 같은 고용 상태 집단의 평균 신용위험점수와 비교했을 때 해당 사용자가 평균보다 얼마나 위험한지를 추가로 볼 수 있습니다.

### make_features()

실제 원본 데이터 프레임에 대해 파생변수를 대량으로 생성하는 함수입니다

1) 주소 관련 피처
현재 주소 유지 기간
이전 주소 유지 기간
두 기간의 합 또는 비율

이런 피처는 주소 변경 패턴이 비정상적으로 보이는 경우 fraud와 관련될 수 있다는 가정에서 만든다.

2) 요청 빈도 / burst 관련 피처
최근 6시간 / 24시간 / 4주 요청량 조합
burst 여부
요청량 비율

짧은 시간 내 과도한 요청은 비정상 거래 패턴일 수 있기 때문에, 이런 피처들이 중요하다.

3) 신원 및 연락처 관련 피처
연락처 유효성
이름-이메일 유사도 기반 위험 피처

실제 사용자 정보와 이메일 정보가 어색하게 연결되어 있는 경우 fraud 가능성을 반영할 수 있다.

4) 기기 관련 피처
기기 사기 이력
기기당 이메일 사용 패턴

하나의 기기에서 여러 계정이 비정상적으로 사용되면 이상 신호일 수 있다.

5) 금융 관련 피처
신용위험점수 관련 변수
한도 대비 이체금액 비율
소득 대비 신용한도 비율

금융적 상식에서 벗어나는 비율은 fraud와 관련될 수 있다.

6) 세션 / 계좌 / 월별 패턴 피처
세션 길이
세션 지속 여부
계좌 관련 조합 변수
월별 패턴

행동 기반 정보도 fraud 탐지에서 중요하므로 함께 반영한다.

7) 복합 위험 점수
여러 위험 신호를 합쳐 risk_score 생성

이건 개별 피처를 따로 쓰는 것뿐 아니라,
여러 위험 요소를 종합한 점수로도 반영하기 위한 것이다.

8) 그룹 평균 피처 추가

앞서 build_grp_stats()에서 만든 그룹별 평균값을 실제 컬럼으로 붙인다.

9) 평균 대비 차이값(diff) 생성

예를 들어:

내 credit_risk_score - 고용 상태별 평균 credit_risk_score
내 req_rate_4w - 주거 상태별 평균 req_rate_4w

이런 식으로 그룹 평균과 얼마나 차이가 나는지도 feature로 만든다.

10) Target Encoding 컬럼 추가

앞서 build_te_map()에서 만든 값을 사용해:

payment_type_te
employment_status_te
housing_status_te
device_os_te
application_source_te

같은 컬럼을 생성한다.

왜 중요한가?
 - 원본 변수만으로는 잘 보이지 않는 fraud 패턴을 파생변수 형태로 더 잘 표현할 수 있습니다

## 셀 4 — 튜닝용 피처 준비

In [4]:
_global_te    = build_te_map(train_df, train_df["fraud_bool"])
_global_grp   = build_grp_stats(train_df)   # 튜닝용은 전체 train 기준 OK
_train_fe     = make_features(train_df, te_map=_global_te, grp_stats=_global_grp)
_X_tune       = _train_fe.drop(columns=["id","fraud_bool"])
_y_tune       = _train_fe["fraud_bool"]
_cat_cols     = _X_tune.select_dtypes(include=["object"]).columns.tolist()
for col in _cat_cols:
    _X_tune[col] = _X_tune[col].fillna("missing")

inf_cnt = np.isinf(_X_tune.select_dtypes(include=[np.number])).sum().sum()
nan_cnt = _X_tune.isnull().sum().sum()
print(f"피처 수: {_X_tune.shape[1]},  inf: {inf_cnt},  NaN: {nan_cnt}")
print("✓ 준비 완료")

피처 수: 98,  inf: 0,  NaN: 10518
✓ 준비 완료


이 셀에서는 방금 정의한 함수들을 사용해서 전체 train 데이터 기준의 feature engineering 결과를 만듭니다

build_te_map(train_df, train_df["fraud_bool"])
전체 train 기준 target encoding map 생성
build_grp_stats(train_df)
전체 train 기준 그룹 평균 통계 생성
make_features(train_df, te_map=..., grp_stats=...)
실제 feature engineering 적용
drop(columns=["id","fraud_bool"])
입력 X에서는 id와 target 제거
_y_tune = fraud_bool
정답 y 분리

그리고 범주형 변수 결측치는 "missing"으로 채우고, inf / NaN 값이 있는지 확인합니다.

왜 필요한가?
 - Optuna 튜닝을 하기 전에, 모델이 바로 학습 가능한 형태의 데이터를 만드는 과정입니다.
   즉, 하이퍼파라미터 탐색용 학습 데이터셋을 만드는 준비단계입니다.

## 셀 5 — CatBoost Optuna 튜닝
★ class_weights + PRAUC + depth 7~10

In [5]:
_cat_file = SAVE_DIR + "best_cat_params_v5.json"

if os.path.exists(_cat_file):
    with open(_cat_file) as f:
        best_cat_params = json.load(f)
    print("✓ 기존 파라미터 로드:", best_cat_params)
else:
    _skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=777)

    def objective(trial):
        cw = trial.suggest_float("class_weight_pos", 3.0, 15.0)
        p  = dict(
            iterations          = trial.suggest_int("iterations", 2000, 5000),
            learning_rate       = trial.suggest_float("learning_rate", 0.005, 0.03, log=True),
            depth               = trial.suggest_int("depth", 7, 10),
            l2_leaf_reg         = trial.suggest_float("l2_leaf_reg", 1, 20),
            bagging_temperature = trial.suggest_float("bagging_temperature", 0.0, 1.0),
            random_strength     = trial.suggest_float("random_strength", 0.1, 5.0),
            border_count        = trial.suggest_int("border_count", 128, 254),
            min_data_in_leaf    = trial.suggest_int("min_data_in_leaf", 3, 30),
        )
        aps = []
        for tr_i, val_i in _skf.split(_X_tune, _y_tune):
            m = CatBoostClassifier(**p,
                loss_function="Logloss",
                eval_metric="PRAUC",
                class_weights=[1.0, cw],
                task_type="CPU", thread_count=-1,
                random_seed=42, verbose=0,
                early_stopping_rounds=150)
            m.fit(_X_tune.iloc[tr_i], _y_tune.iloc[tr_i],
                  cat_features=_cat_cols,
                  eval_set=(_X_tune.iloc[val_i], _y_tune.iloc[val_i]),
                  use_best_model=True)
            aps.append(average_precision_score(
                _y_tune.iloc[val_i], m.predict_proba(_X_tune.iloc[val_i])[:, 1]))
        return np.mean(aps)

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=20, show_progress_bar=True)
    best_cat_params = study.best_params

    with open(_cat_file, "w") as f:
        json.dump(best_cat_params, f, indent=2)
    print(f"✓ 저장 완료 — Best AP: {study.best_value:.5f}")
    print(best_cat_params)

  0%|          | 0/20 [00:00<?, ?it/s]

✓ 저장 완료 — Best AP: 0.17478
{'class_weight_pos': 12.842388843092216, 'iterations': 4247, 'learning_rate': 0.009332139077320492, 'depth': 7, 'l2_leaf_reg': 12.990602527240334, 'bagging_temperature': 0.6171013519037596, 'random_strength': 3.771606178536394, 'border_count': 207, 'min_data_in_leaf': 12}


### CatBoost Optuna 튜닝

best_cat_params_v5.json 파일이 이미 있으면 기존 최적 파라미터를 불러오고, 없으면 Optuna를 사용해 새롭게 하이퍼파라미터 탐색을 수행합니다

탐색 대상 파라미터 :

예를 들어 다음과 같은 값들을 탐색한다.

* iterations
* learning_rate
* depth
* l2_leaf_reg
* bagging_temperature
* random_strength
* border_count
* min_data_in_leaf
* class_weight_pos

평가 방식 : 
* StratifiedKFold(n_splits=3) 사용
* 각 fold마다 CatBoost를 학습
* validation 예측값에 대해 Average Precision 계산
* 3개 fold 평균 AP를 objective 값으로 사용

모델 설정에서 중요한 점
* 불균형 데이터를 보정하기 위해 class_weights를 사용해서 fraud class에 더 큰 가중치 둠

왜 필요한가
 - 모델 성능은 하이퍼파라미터에 따라 크게 달라질 수 있습니다. 따라서 depth, learning rate, iteration 수 등을 자동적으로 탐색해 현재 데이터에 맞느 ㄴ최적 조합을 찾는 과정이 필요함

## 셀 6 — 5-Fold 초기화 + 기존 결과 자동 로드

In [6]:
skf5    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
n_train = len(train_df)
n_test  = len(test_df)

oof_pred  = np.zeros(n_train)
test_pred = np.zeros(n_test)
fold_aps  = []
FOLDS_DONE = []

for _f in range(1, 6):
    _p = SAVE_DIR + f"catboost_v5_fold{_f}.npz"
    if os.path.exists(_p):
        _d = np.load(_p)
        oof_pred[_d["val_idx"]] = _d["oof"]
        test_pred += _d["test"] / 5
        fold_aps.append(float(_d["ap"]))
        FOLDS_DONE.append(_f)
        print(f"✓ Fold {_f} 로드  AP: {float(_d['ap']):.5f}")

print(f"완료: {FOLDS_DONE}  |  남은: {[f for f in range(1,6) if f not in FOLDS_DONE]}")

완료: []  |  남은: [1, 2, 3, 4, 5]


실제 최종 핛브을 위한 5-fold 구조를 준비합니다

* StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
* oof_pred 초기화
* test_pred 초기화
* fold_aps 초기화
* FOLDS_DONE 리스트 생성

그리고 각 fold 결과 파일이 이미 저장되어 있으면 다시 학습을 하지 않고 기존 결과를 불러와 재사용합니다

왜 필요한가?
 - 학습 도중 중단되더라도 다시 이어가기 쉽고, 이미 끝난 fold를 다시 돌리지 않아도 되며 실험 시간을 절약할 수 있습니다.

Fold는 구조가 사실상 동일합니다. 따라서 Fold는 Fold 학습의 가장 아래에 적겠습니다.

## 셀 7 — Fold 1 학습

In [7]:
FOLD = 1
if FOLD in FOLDS_DONE:
    print(f"✓ Fold {FOLD} 이미 완료 — 스킵")
else:
    print(f"==================== Fold {FOLD} ====================")
    _splits = list(skf5.split(train_df, train_df["fraud_bool"]))
    tr_idx, val_idx = _splits[FOLD - 1]

    tr_raw  = train_df.iloc[tr_idx]
    val_raw = train_df.iloc[val_idx]

    # fold-safe TE + 그룹 통계 (train split 기준만)
    fold_te  = build_te_map(tr_raw, tr_raw["fraud_bool"])
    fold_grp = build_grp_stats(tr_raw)

    Xtr_fe  = make_features(tr_raw,  te_map=fold_te, grp_stats=fold_grp)
    Xval_fe = make_features(val_raw, te_map=fold_te, grp_stats=fold_grp)
    Xte_fe  = make_features(test_df, te_map=fold_te, grp_stats=fold_grp)

    ytr  = Xtr_fe["fraud_bool"]
    yval = Xval_fe["fraud_bool"]
    Xtr  = Xtr_fe.drop(columns=["id","fraud_bool"])
    Xval = Xval_fe.drop(columns=["id","fraud_bool"])
    Xte  = Xte_fe.drop(columns=["id"])

    cat_cols = Xtr.select_dtypes(include=["object"]).columns.tolist()
    for col in cat_cols:
        for df_ in [Xtr, Xval, Xte]:
            df_[col] = df_[col].fillna("missing")

    _params = dict(best_cat_params)
    cw = _params.pop("class_weight_pos", 8.0)
    params = {
        **_params,
        "loss_function" : "Logloss",
        "eval_metric"   : "PRAUC",
        "class_weights"  : [1.0, cw],
        "task_type"     : "CPU",
        "thread_count"  : -1,
        "random_seed"   : 42 + FOLD,
        "verbose"       : 200,
    }

    model = CatBoostClassifier(**params)
    model.fit(Xtr, ytr,
              cat_features=cat_cols,
              eval_set=(Xval, yval),
              use_best_model=True,
              early_stopping_rounds=150)

    _oof  = model.predict_proba(Xval)[:, 1]
    _test = model.predict_proba(Xte)[:, 1]
    ap    = average_precision_score(yval, _oof)

    oof_pred[val_idx] = _oof
    test_pred        += _test / 5
    fold_aps.append(ap)
    FOLDS_DONE.append(FOLD)

    np.savez(SAVE_DIR + f"catboost_v5_fold{FOLD}.npz",
             val_idx=val_idx, oof=_oof, test=_test, ap=ap)
    print(f"  CatBoost AP: {ap:.5f}")
    print(f"  ✓ Fold {FOLD} 저장 완료")

==================== Fold 1 ====================
0:	learn: 0.4616988	test: 0.4462110	best: 0.4462110 (0)	total: 273ms	remaining: 19m 20s
200:	learn: 0.5819813	test: 0.5623455	best: 0.5623455 (200)	total: 1m 8s	remaining: 22m 53s
400:	learn: 0.6079028	test: 0.5845889	best: 0.5845889 (400)	total: 2m 17s	remaining: 21m 56s
600:	learn: 0.6221069	test: 0.5943149	best: 0.5943149 (600)	total: 3m 25s	remaining: 20m 43s
800:	learn: 0.6321455	test: 0.6004005	best: 0.6004005 (800)	total: 4m 30s	remaining: 19m 23s
1000:	learn: 0.6400444	test: 0.6041798	best: 0.6041798 (1000)	total: 5m 38s	remaining: 18m 17s
1200:	learn: 0.6464311	test: 0.6068304	best: 0.6068304 (1200)	total: 6m 45s	remaining: 17m 8s
1400:	learn: 0.6534697	test: 0.6087093	best: 0.6087138 (1398)	total: 7m 51s	remaining: 15m 57s
1600:	learn: 0.6642204	test: 0.6114471	best: 0.6114481 (1599)	total: 9m	remaining: 14m 54s
1800:	learn: 0.6761576	test: 0.6137116	best: 0.6137134 (1797)	total: 10m 9s	remaining: 13m 47s
2000:	learn: 0.6879690

## 셀 8 — Fold 2 학습

In [8]:
FOLD = 2
if FOLD in FOLDS_DONE:
    print(f"✓ Fold {FOLD} 이미 완료 — 스킵")
else:
    print(f"==================== Fold {FOLD} ====================")
    _splits = list(skf5.split(train_df, train_df["fraud_bool"]))
    tr_idx, val_idx = _splits[FOLD - 1]

    tr_raw  = train_df.iloc[tr_idx]
    val_raw = train_df.iloc[val_idx]

    # fold-safe TE + 그룹 통계 (train split 기준만)
    fold_te  = build_te_map(tr_raw, tr_raw["fraud_bool"])
    fold_grp = build_grp_stats(tr_raw)

    Xtr_fe  = make_features(tr_raw,  te_map=fold_te, grp_stats=fold_grp)
    Xval_fe = make_features(val_raw, te_map=fold_te, grp_stats=fold_grp)
    Xte_fe  = make_features(test_df, te_map=fold_te, grp_stats=fold_grp)

    ytr  = Xtr_fe["fraud_bool"]
    yval = Xval_fe["fraud_bool"]
    Xtr  = Xtr_fe.drop(columns=["id","fraud_bool"])
    Xval = Xval_fe.drop(columns=["id","fraud_bool"])
    Xte  = Xte_fe.drop(columns=["id"])

    cat_cols = Xtr.select_dtypes(include=["object"]).columns.tolist()
    for col in cat_cols:
        for df_ in [Xtr, Xval, Xte]:
            df_[col] = df_[col].fillna("missing")

    _params = dict(best_cat_params)
    cw = _params.pop("class_weight_pos", 8.0)
    params = {
        **_params,
        "loss_function" : "Logloss",
        "eval_metric"   : "PRAUC",
        "class_weights"  : [1.0, cw],
        "task_type"     : "CPU",
        "thread_count"  : -1,
        "random_seed"   : 42 + FOLD,
        "verbose"       : 200,
    }

    model = CatBoostClassifier(**params)
    model.fit(Xtr, ytr,
              cat_features=cat_cols,
              eval_set=(Xval, yval),
              use_best_model=True,
              early_stopping_rounds=150)

    _oof  = model.predict_proba(Xval)[:, 1]
    _test = model.predict_proba(Xte)[:, 1]
    ap    = average_precision_score(yval, _oof)

    oof_pred[val_idx] = _oof
    test_pred        += _test / 5
    fold_aps.append(ap)
    FOLDS_DONE.append(FOLD)

    np.savez(SAVE_DIR + f"catboost_v5_fold{FOLD}.npz",
             val_idx=val_idx, oof=_oof, test=_test, ap=ap)
    print(f"  CatBoost AP: {ap:.5f}")
    print(f"  ✓ Fold {FOLD} 저장 완료")

==================== Fold 2 ====================
0:	learn: 0.4304626	test: 0.3811136	best: 0.3811136 (0)	total: 279ms	remaining: 19m 44s
200:	learn: 0.5759913	test: 0.5828826	best: 0.5828826 (200)	total: 1m 7s	remaining: 22m 39s
400:	learn: 0.6029932	test: 0.6044736	best: 0.6044736 (400)	total: 2m 13s	remaining: 21m 18s
600:	learn: 0.6174087	test: 0.6136838	best: 0.6136838 (600)	total: 3m 19s	remaining: 20m 8s
800:	learn: 0.6268084	test: 0.6188422	best: 0.6188422 (800)	total: 4m 22s	remaining: 18m 50s
1000:	learn: 0.6348374	test: 0.6225587	best: 0.6225587 (1000)	total: 5m 27s	remaining: 17m 41s
1200:	learn: 0.6415982	test: 0.6251548	best: 0.6251766 (1198)	total: 6m 31s	remaining: 16m 33s
1400:	learn: 0.6486573	test: 0.6274277	best: 0.6274277 (1400)	total: 7m 35s	remaining: 15m 25s
1600:	learn: 0.6586867	test: 0.6301239	best: 0.6301239 (1600)	total: 8m 45s	remaining: 14m 28s
1800:	learn: 0.6709484	test: 0.6327961	best: 0.6328075 (1796)	total: 9m 58s	remaining: 13m 32s
2000:	learn: 0.682

## 셀 9 — Fold 3 학습

In [9]:
FOLD = 3
if FOLD in FOLDS_DONE:
    print(f"✓ Fold {FOLD} 이미 완료 — 스킵")
else:
    print(f"==================== Fold {FOLD} ====================")
    _splits = list(skf5.split(train_df, train_df["fraud_bool"]))
    tr_idx, val_idx = _splits[FOLD - 1]

    tr_raw  = train_df.iloc[tr_idx]
    val_raw = train_df.iloc[val_idx]

    # fold-safe TE + 그룹 통계 (train split 기준만)
    fold_te  = build_te_map(tr_raw, tr_raw["fraud_bool"])
    fold_grp = build_grp_stats(tr_raw)

    Xtr_fe  = make_features(tr_raw,  te_map=fold_te, grp_stats=fold_grp)
    Xval_fe = make_features(val_raw, te_map=fold_te, grp_stats=fold_grp)
    Xte_fe  = make_features(test_df, te_map=fold_te, grp_stats=fold_grp)

    ytr  = Xtr_fe["fraud_bool"]
    yval = Xval_fe["fraud_bool"]
    Xtr  = Xtr_fe.drop(columns=["id","fraud_bool"])
    Xval = Xval_fe.drop(columns=["id","fraud_bool"])
    Xte  = Xte_fe.drop(columns=["id"])

    cat_cols = Xtr.select_dtypes(include=["object"]).columns.tolist()
    for col in cat_cols:
        for df_ in [Xtr, Xval, Xte]:
            df_[col] = df_[col].fillna("missing")

    _params = dict(best_cat_params)
    cw = _params.pop("class_weight_pos", 8.0)
    params = {
        **_params,
        "loss_function" : "Logloss",
        "eval_metric"   : "PRAUC",
        "class_weights"  : [1.0, cw],
        "task_type"     : "CPU",
        "thread_count"  : -1,
        "random_seed"   : 42 + FOLD,
        "verbose"       : 200,
    }

    model = CatBoostClassifier(**params)
    model.fit(Xtr, ytr,
              cat_features=cat_cols,
              eval_set=(Xval, yval),
              use_best_model=True,
              early_stopping_rounds=150)

    _oof  = model.predict_proba(Xval)[:, 1]
    _test = model.predict_proba(Xte)[:, 1]
    ap    = average_precision_score(yval, _oof)

    oof_pred[val_idx] = _oof
    test_pred        += _test / 5
    fold_aps.append(ap)
    FOLDS_DONE.append(FOLD)

    np.savez(SAVE_DIR + f"catboost_v5_fold{FOLD}.npz",
             val_idx=val_idx, oof=_oof, test=_test, ap=ap)
    print(f"  CatBoost AP: {ap:.5f}")
    print(f"  ✓ Fold {FOLD} 저장 완료")

==================== Fold 3 ====================
0:	learn: 0.3969220	test: 0.3899251	best: 0.3899251 (0)	total: 251ms	remaining: 17m 46s
200:	learn: 0.5771662	test: 0.5699994	best: 0.5699994 (200)	total: 1m 7s	remaining: 22m 45s
400:	learn: 0.6049838	test: 0.5930743	best: 0.5930743 (400)	total: 2m 15s	remaining: 21m 42s
600:	learn: 0.6195127	test: 0.6029771	best: 0.6029771 (600)	total: 3m 21s	remaining: 20m 24s
800:	learn: 0.6297637	test: 0.6084610	best: 0.6084610 (800)	total: 4m 26s	remaining: 19m 4s
1000:	learn: 0.6375976	test: 0.6120360	best: 0.6120360 (1000)	total: 5m 30s	remaining: 17m 50s
1200:	learn: 0.6443978	test: 0.6144096	best: 0.6144096 (1200)	total: 6m 35s	remaining: 16m 41s
1400:	learn: 0.6517473	test: 0.6165421	best: 0.6165457 (1399)	total: 7m 40s	remaining: 15m 34s
1600:	learn: 0.6620009	test: 0.6195189	best: 0.6195189 (1600)	total: 8m 48s	remaining: 14m 33s
1800:	learn: 0.6744859	test: 0.6214883	best: 0.6214914 (1799)	total: 9m 58s	remaining: 13m 33s
2000:	learn: 0.685

## 셀 10 — Fold 4 학습

In [10]:
FOLD = 4
if FOLD in FOLDS_DONE:
    print(f"✓ Fold {FOLD} 이미 완료 — 스킵")
else:
    print(f"==================== Fold {FOLD} ====================")
    _splits = list(skf5.split(train_df, train_df["fraud_bool"]))
    tr_idx, val_idx = _splits[FOLD - 1]

    tr_raw  = train_df.iloc[tr_idx]
    val_raw = train_df.iloc[val_idx]

    # fold-safe TE + 그룹 통계 (train split 기준만)
    fold_te  = build_te_map(tr_raw, tr_raw["fraud_bool"])
    fold_grp = build_grp_stats(tr_raw)

    Xtr_fe  = make_features(tr_raw,  te_map=fold_te, grp_stats=fold_grp)
    Xval_fe = make_features(val_raw, te_map=fold_te, grp_stats=fold_grp)
    Xte_fe  = make_features(test_df, te_map=fold_te, grp_stats=fold_grp)

    ytr  = Xtr_fe["fraud_bool"]
    yval = Xval_fe["fraud_bool"]
    Xtr  = Xtr_fe.drop(columns=["id","fraud_bool"])
    Xval = Xval_fe.drop(columns=["id","fraud_bool"])
    Xte  = Xte_fe.drop(columns=["id"])

    cat_cols = Xtr.select_dtypes(include=["object"]).columns.tolist()
    for col in cat_cols:
        for df_ in [Xtr, Xval, Xte]:
            df_[col] = df_[col].fillna("missing")

    _params = dict(best_cat_params)
    cw = _params.pop("class_weight_pos", 8.0)
    params = {
        **_params,
        "loss_function" : "Logloss",
        "eval_metric"   : "PRAUC",
        "class_weights"  : [1.0, cw],
        "task_type"     : "CPU",
        "thread_count"  : -1,
        "random_seed"   : 42 + FOLD,
        "verbose"       : 200,
    }

    model = CatBoostClassifier(**params)
    model.fit(Xtr, ytr,
              cat_features=cat_cols,
              eval_set=(Xval, yval),
              use_best_model=True,
              early_stopping_rounds=150)

    _oof  = model.predict_proba(Xval)[:, 1]
    _test = model.predict_proba(Xte)[:, 1]
    ap    = average_precision_score(yval, _oof)

    oof_pred[val_idx] = _oof
    test_pred        += _test / 5
    fold_aps.append(ap)
    FOLDS_DONE.append(FOLD)

    np.savez(SAVE_DIR + f"catboost_v5_fold{FOLD}.npz",
             val_idx=val_idx, oof=_oof, test=_test, ap=ap)
    print(f"  CatBoost AP: {ap:.5f}")
    print(f"  ✓ Fold {FOLD} 저장 완료")

==================== Fold 4 ====================
0:	learn: 0.4115179	test: 0.3904913	best: 0.3904913 (0)	total: 263ms	remaining: 18m 37s
200:	learn: 0.5790926	test: 0.5457175	best: 0.5457175 (200)	total: 1m 7s	remaining: 22m 35s
400:	learn: 0.6088217	test: 0.5723336	best: 0.5723336 (400)	total: 2m 15s	remaining: 21m 40s
600:	learn: 0.6238903	test: 0.5835274	best: 0.5835274 (600)	total: 3m 22s	remaining: 20m 29s
800:	learn: 0.6345630	test: 0.5901832	best: 0.5901832 (800)	total: 4m 29s	remaining: 19m 21s
1000:	learn: 0.6417646	test: 0.5940266	best: 0.5940266 (1000)	total: 5m 32s	remaining: 17m 56s
1200:	learn: 0.6481166	test: 0.5969251	best: 0.5969251 (1200)	total: 6m 34s	remaining: 16m 41s
1400:	learn: 0.6551738	test: 0.5994535	best: 0.5994580 (1399)	total: 7m 39s	remaining: 15m 33s
1600:	learn: 0.6654949	test: 0.6022362	best: 0.6022362 (1600)	total: 8m 48s	remaining: 14m 33s
1800:	learn: 0.6779102	test: 0.6039658	best: 0.6039658 (1800)	total: 9m 58s	remaining: 13m 32s
2000:	learn: 0.68

## 셀 11 — Fold 5 학습

In [11]:
FOLD = 5
if FOLD in FOLDS_DONE:
    print(f"✓ Fold {FOLD} 이미 완료 — 스킵")
else:
    print(f"==================== Fold {FOLD} ====================")
    _splits = list(skf5.split(train_df, train_df["fraud_bool"]))
    tr_idx, val_idx = _splits[FOLD - 1]

    tr_raw  = train_df.iloc[tr_idx]
    val_raw = train_df.iloc[val_idx]

    # fold-safe TE + 그룹 통계 (train split 기준만)
    fold_te  = build_te_map(tr_raw, tr_raw["fraud_bool"])
    fold_grp = build_grp_stats(tr_raw)

    Xtr_fe  = make_features(tr_raw,  te_map=fold_te, grp_stats=fold_grp)
    Xval_fe = make_features(val_raw, te_map=fold_te, grp_stats=fold_grp)
    Xte_fe  = make_features(test_df, te_map=fold_te, grp_stats=fold_grp)

    ytr  = Xtr_fe["fraud_bool"]
    yval = Xval_fe["fraud_bool"]
    Xtr  = Xtr_fe.drop(columns=["id","fraud_bool"])
    Xval = Xval_fe.drop(columns=["id","fraud_bool"])
    Xte  = Xte_fe.drop(columns=["id"])

    cat_cols = Xtr.select_dtypes(include=["object"]).columns.tolist()
    for col in cat_cols:
        for df_ in [Xtr, Xval, Xte]:
            df_[col] = df_[col].fillna("missing")

    _params = dict(best_cat_params)
    cw = _params.pop("class_weight_pos", 8.0)
    params = {
        **_params,
        "loss_function" : "Logloss",
        "eval_metric"   : "PRAUC",
        "class_weights"  : [1.0, cw],
        "task_type"     : "CPU",
        "thread_count"  : -1,
        "random_seed"   : 42 + FOLD,
        "verbose"       : 200,
    }

    model = CatBoostClassifier(**params)
    model.fit(Xtr, ytr,
              cat_features=cat_cols,
              eval_set=(Xval, yval),
              use_best_model=True,
              early_stopping_rounds=150)

    _oof  = model.predict_proba(Xval)[:, 1]
    _test = model.predict_proba(Xte)[:, 1]
    ap    = average_precision_score(yval, _oof)

    oof_pred[val_idx] = _oof
    test_pred        += _test / 5
    fold_aps.append(ap)
    FOLDS_DONE.append(FOLD)

    np.savez(SAVE_DIR + f"catboost_v5_fold{FOLD}.npz",
             val_idx=val_idx, oof=_oof, test=_test, ap=ap)
    print(f"  CatBoost AP: {ap:.5f}")
    print(f"  ✓ Fold {FOLD} 저장 완료")

==================== Fold 5 ====================
0:	learn: 0.4354372	test: 0.4191554	best: 0.4191554 (0)	total: 271ms	remaining: 19m 10s
200:	learn: 0.5801606	test: 0.5727820	best: 0.5727820 (200)	total: 1m 7s	remaining: 22m 44s
400:	learn: 0.6069902	test: 0.5918552	best: 0.5918552 (400)	total: 2m 16s	remaining: 21m 51s
600:	learn: 0.6218942	test: 0.6009489	best: 0.6009489 (600)	total: 3m 23s	remaining: 20m 33s
800:	learn: 0.6320931	test: 0.6060645	best: 0.6060645 (800)	total: 4m 29s	remaining: 19m 19s
1000:	learn: 0.6392191	test: 0.6092168	best: 0.6092243 (999)	total: 5m 34s	remaining: 18m 4s
1200:	learn: 0.6462407	test: 0.6115210	best: 0.6115210 (1200)	total: 6m 39s	remaining: 16m 52s
1400:	learn: 0.6531901	test: 0.6134186	best: 0.6134186 (1400)	total: 7m 47s	remaining: 15m 49s
1600:	learn: 0.6632652	test: 0.6159007	best: 0.6159007 (1600)	total: 8m 55s	remaining: 14m 44s
1800:	learn: 0.6756724	test: 0.6180511	best: 0.6180683 (1799)	total: 10m 5s	remaining: 13m 42s
2000:	learn: 0.6874

### Fold

1) train / validation split 분리

skf5.split(train_df, train_df["fraud_bool"])를 이용해서
해당 fold의 train index와 validation index를 나눈다.

2) raw 데이터 분리
* tr_raw
* val_raw
로 분리한다.

3) fold 기준 target encoding / 그룹 통계 재생성

여기서 매우 중요한 작업이 들어간다.

* fold_te = build_te_map(tr_raw, tr_raw["fraud_bool"])
* fold_grp = build_grp_stats(tr_raw)

즉, 전체 데이터가 아니라 fold의 train 부분만 가지고
target encoding과 그룹 통계를 다시 만든다.

4) train / val / test에 동일한 feature engineering 적용
* Xtr_fe
* Xval_fe
* Xte_fe

를 만든다.

여기서 validation과 test에도 같은 feature 구조를 적용하지만,
기준이 되는 통계값은 fold train에서만 계산된 것이다.

5) 입력/정답 분리
* 학습 입력 X
* validation 입력 X
* 타깃 y
를 분리한다.
6) CatBoost 학습

앞서 찾은 best_cat_params를 사용해 CatBoost 모델을 학습한다.

7) validation 예측 및 AP 계산

validation set에서 예측을 수행하고 Average Precision을 계산한다.
이 값이 해당 fold의 성능이다.

8) test 예측 저장

각 fold 모델이 test set에 대해 예측한 값을 저장한다.

9) fold 결과 저장

해당 fold에서 나온:

* validation 예측
* test 예측
* AP score
등을 파일로 저장해 재사용 가능하게 만든다.

왜 이렇게 하는가?

여기서 핵심은 데이터 누수(leakage) 방지다.

만약 target encoding이나 그룹 평균 통계를
validation 데이터까지 포함해서 만들어버리면,
모델이 검증 데이터를 간접적으로 미리 본 셈이 되어 성능이 부풀려질 수 있다.

그래서 각 fold마다:

train 부분만으로 통계를 계산하고
그 통계를 validation/test에 적용하는 방식으로 설계한 것이다.

## 셀 12 — 최종 결과 + 제출 파일

In [12]:
if len(FOLDS_DONE) < 5:
    print(f"⚠ 완료된 Fold: {FOLDS_DONE}")
else:
    y_all  = train_df["fraud_bool"].values
    oof_ap = average_precision_score(y_all, oof_pred)

    print("=" * 45)
    print(f"Fold APs : {[f'{s:.5f}' for s in fold_aps]}")
    print(f"Mean AP  : {np.mean(fold_aps):.5f}")
    print(f"OOF AP   : {oof_ap:.5f}")
    print("=" * 45)

    submission = sub.copy()
    submission["fraud"] = test_pred
    out = SAVE_DIR + "submission_v5.csv"
    submission.to_csv(out, index=False)
    print(f"✓ 저장: {out}")
    print(submission["fraud"].describe())

Fold APs : ['0.17456', '0.18491', '0.17926', '0.16964', '0.17458']
Mean AP  : 0.17659
OOF AP   : 0.17576
✓ 저장: Desktop\cuaida\saves\\submission_v5.csv
count    300000.000000
mean          0.078828
std           0.127443
min           0.000089
25%           0.010836
50%           0.029166
75%           0.083579
max           0.988489
Name: fraud, dtype: float64


모든 fold 학습이 끝난 후 최종 결과를 정리합니다.

* oof_ap = average_precision_score(y_all, oof_pred)
* fold별 AP 출력
* 평균 AP 출력
* 전체 OOF AP 출력

그리고 마지막으로:

* 각 fold의 test 예측값을 평균내어
* submission 형식에 맞춰 저장한다.

OOF가 의미하는 것

OOF(Out-of-Fold) 예측은 각 fold의 validation 예측을 전체 train 기준으로 합친 값이다.
즉, train 전체에 대해 “각 샘플이 자신이 학습에 포함되지 않은 모델로 예측된 결과”라고 볼 수 있다.

그래서 OOF AP는
단순 fold 평균보다도 전체적인 일반화 성능을 보는 데 유용하다.

왜 필요한가?

이 단계는:

모델 성능을 최종적으로 확인하고
제출 가능한 결과 파일을 생성하는
마지막 마무리 단계다.

## 요약

전체 코드 흐름은 다음과 같습니다.

1. 환경 설정 및 라이브러리 로드
   첫 번째 셀에서는 pandas, numpy, CatBoost, Optuna, StratifiedKFold, average_precision_score 등의 라이브러리를 불러오고, 데이터 및 결과 저장 경로를 설정합니다. 이후 모델 학습과 평가에 필요한 기본 환경을 준비하는 단계입니다.

2. 데이터 로드
   두 번째 셀에서는 train.csv, test.csv, sample_submission.csv를 불러오고, 데이터 shape와 fraud 비율을 확인합니다. 이를 통해 데이터가 정상적으로 로드되었는지와 클래스 불균형 여부를 점검합니다.

3. Feature Engineering 함수 정의
   세 번째 셀은 핵심 셀로, fraud 탐지에 유의미한 파생변수를 생성하는 함수들이 정의되어 있습니다.

* `build_te_map()` : 주요 범주형 변수에 대해 target encoding 값을 생성
* `build_grp_stats()` : 그룹별 평균 통계 계산
* `make_features()` : 주소 관련, 요청 빈도, 금융 비율, 기기 이력, 복합 위험 점수, 그룹 평균 및 차이값, target encoding 컬럼 등을 포함한 최종 파생변수를 생성

4. 튜닝용 feature 준비
   네 번째 셀에서는 전체 train 데이터 기준으로 target encoding과 그룹 통계를 생성한 뒤 feature engineering을 적용하여 Optuna 튜닝용 입력 데이터와 정답 데이터를 구성합니다. 이 과정에서 id, fraud_bool 분리 및 결측치/비정상값 점검도 수행합니다.

5. CatBoost Optuna 튜닝
   다섯 번째 셀에서는 Optuna를 사용해 CatBoost의 depth, learning_rate, iterations, class_weight_pos 등 주요 하이퍼파라미터를 탐색합니다. 평가 기준은 Average Precision이며, 3-fold 교차검증 평균 성능이 가장 좋은 파라미터를 선택합니다.

6. 5-Fold 초기화
   여섯 번째 셀에서는 StratifiedKFold(n_splits=5)를 설정하고, OOF 예측값, test 예측값, fold별 AP를 저장할 배열을 초기화합니다. 또한 이미 저장된 fold 결과가 있으면 재사용할 수 있도록 구성했습니다.

7. Fold별 학습
   일곱 번째 셀부터 열한 번째 셀까지는 Fold 1~5 학습 셀입니다. 각 fold에서 train/validation을 분리한 뒤, train split 기준으로만 target encoding과 그룹 통계를 다시 생성하여 leakage를 방지합니다. 이후 동일한 feature engineering을 train/validation/test에 적용하고, 최적 파라미터 기반 CatBoost를 학습한 뒤 validation AP와 test 예측값을 저장합니다.

8. 최종 결과 정리 및 제출 파일 생성
   마지막 셀에서는 fold별 AP, 평균 AP, 전체 OOF AP를 계산하고, 각 fold의 test 예측값 평균을 이용해 최종 submission 파일을 생성합니다.

전체적으로는 원본 데이터를 불러온 뒤 fraud 관련 패턴을 더 잘 드러내는 파생변수를 생성하고, CatBoost를 Optuna로 튜닝한 후 5-fold 교차검증으로 안정적으로 학습하여 최종 예측을 만드는 구조입니다.


## 예상 질문

1. 왜 feature engineering을 많이 했는가

fraud 패턴은 원본 변수 하나만으로 드러나지 않는 경우가 많아서, 주소/요청 빈도/금융 비율/기기 이력 등의 정보를 조합한 파생변수를 통해 위험 신호를 더 잘 표현하고자 했습니다.

2. 왜 target encoding을 썼는가

주요 범주형 변수의 fraud 경향을 반영하기 위해 단순 문자열 대신 target encoding을 적용했습니다.

3. 왜 CatBoost를 썼는가

범주형 변수와 tabular 데이터에 강점이 있고, 불균형 데이터 환경에서도 안정적인 성능을 기대할 수 있기 때문입니다.

4. 왜 fold마다 통계를 다시 만들었는가

validation 정보가 통계 계산에 섞이면 데이터 누수가 발생할 수 있기 때문에, 각 fold의 train 데이터만으로 target encoding과 그룹 통계를 다시 계산했습니다.